<a id="top"></a>

# Demo: the scratchpad is an interface contract

```yaml
title: "Demo: the scratchpad is an interface contract"
keywords: scratchpad, thinking tags, answer tags, defensive parsing, interface contract, teacher demo
estimated duration: 12
```

> **Lesson:** L06. Teacher demo — Demo 2 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L06/demos_or_activities.md). Makes **live** Claude calls through `potato_llm` — set `ANTHROPIC_API_KEY` before running. Clear outputs before committing.
>
> The point to land: `<thinking>` tags add **zero capability** — they are a contract about *shape*, the same move as L02's structured output. The parser is the enforcement.

## Contents

- [1. Setup](#1-setup)
- [2. Reason inside the tags, answer outside](#2-reason-inside-the-tags-answer-outside)
- [3. The contract is best-effort — parse defensively](#3-the-contract-is-best-effort--parse-defensively)
- [4. Takeaways](#4-takeaways)

## 1. Setup

Reuse [Demo 1](L0603_lecture.ipynb)'s problem so the technique stacks on something familiar, plus a tiny defensive tag parser.

In [ ]:
import re

from fluffy_potato_curriculum.potato_llm import AnthropicClient, Message, PotatoLLMClient

PROBLEM = (
    "A bag contains 5 red, 7 blue, and 3 green marbles. I draw 3 without replacement. "
    "What is the probability all three are different colors?"
)

TAGGED_PROMPT = (
    PROBLEM
    + " Put your reasoning inside <thinking>...</thinking> tags, "
    + "and put ONLY your final numeric answer inside <answer>...</answer> tags."
)


def extract_tag(text: str, tag: str) -> str | None:
    """Pull the contents of <tag>...</tag>, or None if the contract was broken."""
    match = re.search(rf"<{tag}>(.*?)</{tag}>", text, re.DOTALL)
    return match.group(1).strip() if match else None


client: PotatoLLMClient = AnthropicClient()
print(f"setup ready (live client: {client.model})")

[↑ Back to top](#top)

## 2. Reason inside the tags, answer outside

The model already reasoned in Demo 1. Here the same reasoning is wrapped for parsing.

In [ ]:
reply = client.chat([Message.user(TAGGED_PROMPT)], max_tokens=600, temperature=0.0)
print("raw reply:\n", reply.text)

answer = extract_tag(reply.text, "answer")
thinking = extract_tag(reply.text, "thinking")
print("\n--- parsed answer (what your code consumes) ---")
print(answer)
print("\n--- thinking is right there for debugging, but your program ignores it ---")
print((thinking or "")[:200], "...")

[↑ Back to top](#top)

## 3. The contract is best-effort — parse defensively

The model **agreed** to the tags; it did not **guarantee** them. A real run is usually clean, so here is a crafted broken reply (answer outside the tags) to show what your parser must survive. Fail **loudly**, never silently.

In [ ]:
broken = (
    "<thinking>5/12 times 7/11 times ... I will just say it below.</thinking>\n"
    "The probability is about 0.23."  # answer NOT inside <answer> tags
)
salvaged = extract_tag(broken, "answer")
if salvaged is None:
    print('CONTRACT BROKEN: no <answer> block. Fall back or fail loudly — do NOT return "".')
    print("raw reply was:", repr(broken))
else:
    print("answer:", salvaged)

[↑ Back to top](#top)

## 4. Takeaways

- The tags add **no capability** — the model reasoned identically in Demo 1 without them. They add a *boundary* for downstream code.
- Same discipline as L02's JSON parsing: ask for the shape, then **enforce it in code** and fail loudly on a violation.
- This is what L11 (tracing) and L12 (evaluation) score against: a clean answer field, with the reasoning logged separately for debugging.

[↑ Back to top](#top)